# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


In [1]:
from pathlib import Path
from roboflow import Roboflow
from api_key import load_api_key

WORKSPACE = "anbies-workspace"
PROJECT = "egg-quality-detection-nvhwd"
VERSION = 5
DATA_DIR = Path("datasets") / f"{PROJECT}-v{VERSION}"

if not (DATA_DIR / "data.yaml").exists():
    rf = Roboflow(api_key=load_api_key())
    version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
    version.download("yolov11", location=str(DATA_DIR))

DATA_YAML = DATA_DIR / "data.yaml"
print("Dataset ready at", DATA_YAML)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to datasets/egg-quality-detection-nvhwd-v5 in yolov11:: 100%|██████████| 20167/20167 [00:02<00:00, 8292.15it/s]

Dataset ready at datasets/egg-quality-detection-nvhwd-v5/data.yaml


In [2]:
import json
import numpy as np
import yaml
from collections import defaultdict

cfg = yaml.safe_load(open(DATA_YAML))
names = cfg["names"]
names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}

stats = {}
for split in ["train", "valid", "test"]:
    n_images = len(list((DATA_DIR / split / "images").glob("*")))
    counts = defaultdict(int)
    areas = defaultdict(list)   # box area as a fraction of the image
    empty = 0
    per_image = []
    for f in (DATA_DIR / split / "labels").glob("*.txt"):
        rows = [l.split() for l in f.read_text().splitlines() if l.strip()]
        if not rows:
            empty += 1
        per_image.append(len(rows))
        for r in rows:
            c = int(r[0])
            counts[names[c]] += 1
            areas[names[c]].append(float(r[3]) * float(r[4]))  # w * h, already normalised
    stats[split] = {
        "images": n_images,
        "images_without_labels": empty,
        "mean_objects_per_image": float(np.mean(per_image)) if per_image else 0,
        "instances": dict(counts),
        "median_area_pct": {k: round(100 * float(np.median(v)), 2) for k, v in areas.items()},
        "p10_p90_area_pct": {k: [round(100 * float(np.percentile(v, 10)), 2),
                                 round(100 * float(np.percentile(v, 90)), 2)] for k, v in areas.items()},
    }

Path("results").mkdir(exist_ok=True)
json.dump(stats, open("results/dataset_stats.json", "w"), indent=2)
for split, s in stats.items():
    print(split, json.dumps(s, indent=2))

train {
  "images": 8822,
  "images_without_labels": 426,
  "mean_objects_per_image": 0.95273180684652,
  "instances": {
    "good": 4208,
    "cracked": 4197
  },
  "median_area_pct": {
    "good": 20.9,
    "cracked": 49.22
  },
  "p10_p90_area_pct": {
    "good": [
      10.09,
      33.14
    ],
    "cracked": [
      24.36,
      76.61
    ]
  }
}
valid {
  "images": 639,
  "images_without_labels": 39,
  "mean_objects_per_image": 0.9405320813771518,
  "instances": {
    "cracked": 307,
    "good": 294
  },
  "median_area_pct": {
    "cracked": 43.3,
    "good": 18.94
  },
  "p10_p90_area_pct": {
    "cracked": [
      21.59,
      61.1
    ],
    "good": [
      9.39,
      25.83
    ]
  }
}
test {
  "images": 620,
  "images_without_labels": 20,
  "mean_objects_per_image": 0.967741935483871,
  "instances": {
    "cracked": 297,
    "good": 303
  },
  "median_area_pct": {
    "cracked": 43.42,
    "good": 19.06
  },
  "p10_p90_area_pct": {
    "cracked": [
      22.06,
      65.05


In [ ]:
import json, shutil
import torch, yaml
from collections import Counter
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Training on:", device)

cfg = yaml.safe_load(open(DATA_YAML))
ABS_YAML = DATA_DIR.resolve() / "data_abs.yaml"
yaml.safe_dump({
    "path": str(DATA_DIR.resolve()),
    "train": "train/images", "val": "valid/images", "test": "test/images",
    "names": cfg["names"],
}, open(ABS_YAML, "w"))

RUNS = str(Path("runs").resolve())
model = YOLO("yolo11n.pt")           # pretrained COCO checkpoint, finetuned below
model.train(
    data=str(ABS_YAML), epochs=30, imgsz=640, batch=16,
    patience=10, seed=0, device=device,
    project=RUNS, name="egg", exist_ok=True,
)

Path("weights").mkdir(exist_ok=True)
shutil.copy(Path(model.trainer.best), "weights/best.pt")   # best epoch picked on the valid split

In [ ]:
def summarize(m):
    per = {}
    for i, idx in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        per[names[int(idx)]] = dict(precision=float(p), recall=float(r),
                                    ap50=float(ap50), ap50_95=float(ap))
    return dict(map50=float(m.box.map50), map50_95=float(m.box.map), per_class=per)

val_kwargs = dict(data=str(ABS_YAML), split="test", imgsz=640, device=device, project=RUNS, exist_ok=True)

# 1) Finetuned model
m = YOLO("weights/best.pt").val(name="test_finetuned", plots=True, **val_kwargs)
finetuned = summarize(m)
labels = [names[i] for i in sorted(names)] + ["background"]
finetuned["confusion_matrix"] = {"labels": labels, "matrix": m.confusion_matrix.matrix.tolist()}

# 2) Pretrained COCO model, zero-shot
coco = YOLO("yolo11n.pt")
try:
    coco_res = summarize(coco.val(name="test_coco", plots=False, **val_kwargs))
except Exception as e:
    print("COCO val failed:", e)
    coco_res = {"map50": 0.0, "map50_95": 0.0, "per_class": {}}

# What does COCO actually see in these images?
seen, imgs_with_det, n_imgs = Counter(), 0, 0
for r in coco.predict(str(DATA_DIR / "test" / "images"), conf=0.25, stream=True, verbose=False, device=device):
    n_imgs += 1
    if len(r.boxes):
        imgs_with_det += 1
    for c in r.boxes.cls.tolist():
        seen[coco.names[int(c)]] += 1
coco_res["images_with_any_detection"] = f"{imgs_with_det}/{n_imgs}"
coco_res["classes_it_predicts"] = dict(seen.most_common(8))

# 3) Trivial baselines
cracked_id = [k for k, v in names.items() if v == "cracked"][0]
n_with_cracked = sum(
    any(l.split()[0] == str(cracked_id) for l in f.read_text().splitlines() if l.strip())
    for f in (DATA_DIR / "test" / "labels").glob("*.txt")
)
trivial = {
    "predict_nothing": {"map50": 0.0, "map50_95": 0.0, "recall": 0.0},
    "test_images_with_a_cracked_egg": n_with_cracked,
    "test_images": n_imgs,
}

Path("results").mkdir(exist_ok=True)
json.dump({"finetuned": finetuned, "coco_zero_shot": coco_res, "trivial": trivial},
          open("results/metrics.json", "w"), indent=2)
print(json.dumps({"finetuned": finetuned, "coco": coco_res, "trivial": trivial}, indent=2))

In [ ]:
# Raw test predictions at very low confidence, for the cost sweep (CPU-only later)
best = YOLO("weights/best.pt")
gt_dir = DATA_DIR / "test" / "labels"
preds = []
for r in best.predict(str(DATA_DIR / "test" / "images"), conf=0.001, iou=0.6,
                      max_det=50, stream=True, verbose=False, device=device):
    stem = Path(r.path).stem
    f = gt_dir / f"{stem}.txt"
    gt = [[int(l.split()[0]), *map(float, l.split()[1:5])]
          for l in f.read_text().splitlines() if l.strip()] if f.exists() else []
    preds.append({
        "image": stem,
        "gt": gt,   # [class, x, y, w, h], normalised
        "pred": [[int(c), float(s), *b] for c, s, b in
                 zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xywhn.tolist())],
    })
json.dump(preds, open("results/test_predictions.json", "w"))
print(len(preds), "test images saved")